# CrossClientRank projection diagnostic
Owner locked .25h/seed; same weights/sample/basis/rcond/gates. No SGD or utility. Private inline W&B key persists in private version history.

In [ ]:
import time
SESSION_T0=time.monotonic()

import os, subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','--quiet','wandb==0.29.0'],check=True)
import wandb
_INLINE_WANDB_KEY = ""
try:
    from kaggle_secrets import UserSecretsClient
    _wandb_key=_INLINE_WANDB_KEY or UserSecretsClient().get_secret('wandb_key')
except Exception:
    _wandb_key=_INLINE_WANDB_KEY
assert _wandb_key, 'W&B key required before training'
assert wandb.login(key=_wandb_key,anonymous='never',relogin=True,verify=True)
os.environ['WANDB_API_KEY']=_wandb_key
del _wandb_key
assert wandb.Api(timeout=30).default_entity, 'W&B online authentication failed'
print('W&B authenticated online',flush=True)


In [ ]:
MH=['v5_books.py', 'v3_methods.py', 'v4_mala.py', 'v7_nystrom.py', 'v9_anchor.py', 'v8_run.py', 'teacher_driver.py', 'v9_run.py', 'v5_run.py', 'new_methods.py', 'prepare.py', 'archs.py', 'v6_schema.py', '__init__.py', 'channel.py', 'v3_dense.py', 'v10_run.py', 'v6_run.py', 'dagsnet.py', 'metrics.py', 'v4_schema.py', 'new_fl.py', 'v7_run.py', 'model_packet.py', 'v6_geometry.py', 'v3_run.py', 'v8_hybrid.py', 'mobility.py', 'p2.py', 'pilot.py', 'v7_basis.py', 'v2_run.py', 'v3_math.py', 'v5_anchor.py', 'v4_run.py', 'v10_energy.py', 'v4_support.py', 'v2_methods.py', 'v2_math.py', 'teacher_base.py']
PROJ=['rank_math.py', 'rank_probe.py']
INPUTS={'train_14_y.npy': '0388e591c9ac97341fd0e72b2069f4efbea5c92b9d8e4ddb92a93ed5f68ae810', 'train_13_y.npy': '185c46c74758f05822c151479f657652dbaad38566d89380d5ef5ecaad5c70d6', 'v5_books.py': '1123938877061d903c6ebbb029d6f641ee7422c637181dfc93e3a0a600e7b475', 'train_14_X.npy': '274b93855a5050c9bf56bb6446fd797e436ef4408425bdb69434f9537e5070da', 'v3_methods.py': '8c3faad01d13756bd7e27806a7fbb8d75c1f0d7c5751e54c5b4e37c66e4b0f84', 'v4_mala.py': '7d916530164cb7fa3bc9dadf051c47688327659b0e50f825e05e61ca4f805aea', 'train_0_y.npy': '81d72de22935ae850b5a137cdbeabac90523bcc00a33d585bd379b6fae2e0610', 'train_12_X.npy': 'e297270e4fef9c27f2cecb5b1ad69d458650c5a6f50b9ad65383a5d3ddfe5e35', 'v7_nystrom.py': '114e23dcf2ee15f6083f79fe8456edf58b08aad78d807b0ffce9b91d6dc22713', 'v9_anchor.py': '2d15e9b4f53006658a755b8d75e959eb314967773cb557cb1dbb8137ae6a0688', 'v8_run.py': 'dd362c93df255e901a274e3925a11acc9f7271ccacccd2e53c33789000dd4041', 'rank_math.py': '9e6b481aa31cae55053aa1a261813efdb0f2f5e2783fa34b88b9f54ad3deb3ba', 'train_0_X.npy': '6eefa0cd099186d7bbe1323261326e3ce98c519411909ca5214373196264459c', 'teacher_driver.py': 'b46da25f871c7c0fd6fdd78dde9f541168faade2f825dfe32dd86ffa4e95c968', 'v9_run.py': '2f4f460fef23e6d01315b8b80c89a2f2cf5da132119352cb50d3d15a52204614', 'train_10_y.npy': '0992b94ffd5160830c5ade271c7bf17b4322faac96046f3cd6f134fae3e115d3', 'v5_run.py': '9c257035c4388aaf0c48886b9e83f126bca86c812b7fe8a17843894e0200c71e', 'new_methods.py': 'b306660ea84d6bc42f3dcbbef1977e959487f1db3bd333e6b53ed3612df0d2e3', 'prepare.py': '9dd51acb3c23df4a00cec29f60f1805056d0292cfd62a809f071411550111e2e', 'archs.py': '0c24ae80e1ff5401412ea9642c9f168843c965b0dcc36942c5e85b658d9666bd', 'v6_schema.py': '46cf470ef781f5ecf9007e96be2fe0b0bd446bce49e4727fc7df19fe0ee6b654', '__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'train_1_y.npy': 'bd68c140cb5e594f23c86c90166cfe080e1a36b45a2ac2c3b27bd4ae678460d2', 'channel.py': '609b114023990e9b7cdf30cf86b18b388e7fac2884b432ef90bc31b6399b0dc2', 'v3_dense.py': '11a1caad91455da4c013a1f48bd3ccc51c56b6cdce90ff3db26ffa9d1e40292e', 'train_7_X.npy': 'f12f40aecf2ad5b28b1bf710986cc517382c106fe7b581d9d5fb47f51b25fa94', 'v10_run.py': '660cc41c51514a49e8afc28f86c98ee65ab45eb45210ef93350e4f9f27902986', 'train_7_y.npy': '1a0cdd31660a9a4365971a4eb28343479004797b44d7c88ceb13289fd9bd377b', 'v6_run.py': 'd71baa60141a36992810f091894ff8ea84acbd99f227dc0a297e15c429a57f9f', 'source_manifest.json': '00fb447676afb708ec0687bec6485348ce3f62543ab282e8ef86d517e9cf5578', 'dagsnet.py': '5bafe8a571934397194da42f263564ca56093547b03d7eb5735d9c0c65944a5c', 'train_19_y.npy': 'b64e7fe679d9156046e823f135955992f869994fa68fb677a4beefaa680ea0f0', 'metrics.py': '8f23006318a246673eec8c566291a5447b53a16483bcab855a9f768d1da2a712', 'v4_schema.py': '0828b4a83a75ed160643e5f0fb7c6ea418e274c8a2f9a73b01d4f3a6dc94555c', 'new_fl.py': '2d5abcb1813c1399cc29b442ed76a6eea93ce03f4600987773bb1bddec954179', 'train_12_y.npy': 'fcd8f1e70e2c0d0bbf4c5a0ec9addc03f72dd9ea88cbd718b5b2f2547a78c26b', 'train_8_y.npy': '653a8ca50081fa0be465cfb22c48b003a205bcad153b106c551d59233f71745b', 'train_2_X.npy': '7ccbb8b77174417f0849869a5fbe627631c9c46b55d60c1abbf8a71a6d9a4cdc', 'protocol.json': '34b348f27e442c5bdc146698fb11aa26ff3634bce36012f22b97d14dadb5df4a', 'v7_run.py': 'd2ca47dadbe17374e639569ddb277b21708031db174944842c19b9736036d428', 'model_packet.py': '3c4fc9485c936b81804865796191f34b0c353ed5acd8fc8bbe9e53b5c65b77bd', 'train_17_y.npy': '61aa1abbd0f517bec9085a17f661ebf8be2dd082376f553a53de442e0c4ca5c6', 'train_16_X.npy': 'e0f07885973ddf63f9ca84e2a03125ef2a155ea58c6ea38c86b4d4c68d8318e9', 'train_11_X.npy': '5cba03a92ec864d1f2141f169e097d26c2d11a434b880b30b3ae9b70c96582cf', 'train_6_X.npy': 'de88ed8dc095576a53b563491216fd70c5ae40aad50be35d58c5b0eef62d0a22', 'v6_geometry.py': 'f277b7631e109be19de998e4055a21c33a94bf2789097a2f36e2f6a2660fd2cb', 'train_8_X.npy': '7b4b01e987c056b9156bbaf50589608ed80e7c08c005e0c77344a8ee30a6a5b8', 'train_5_y.npy': '4c908bb0992159f83096d2d3e55fae2bb7d68fa0e3c758215f9327491bb07110', 'meta.json': '1aeb0c9156c1f7d77694ffbee6f24a3f6abeec0c0dad48c1a45741cdb85fbff4', 'warm-proof.json': 'fc1bb0f7054f8fdd31a4539a0a479eba1c28164d29c1482570962443c5fbbf8c', 'train_10_X.npy': '2ab1fb1fe476235a698c36b2550c6c6f724c9412c06f9f3fc171957fece05af8', 'train_15_X.npy': 'e299020f0819218af6e7c2c80f7db42eb2c3961bea5612b14bea7bac3d22d9e9', 'v3_run.py': '6903adafd39211907b25385dd74ecf3063922252b364d6f172842ace588c778f', 'v8_hybrid.py': '17683b9c566310bdb22cdde361347d6af7332826c8a95589bc61d4a46b755e72', 'mobility.py': '450ae2599f4a4cbeff1a63292d492b7ab080fa8b119aea1cb7438ced7c9369d7', 'p2.py': '62889bb18a1d448f139dc5e1b01480ea444ef41bcc3ee3394568507cecd28145', 'train_17_X.npy': '8e9ce2afadbdd5b68b5f231e4c9ca5264c1121b2246bb089bc3aa8111ebe6ccc', 'train_13_X.npy': 'd3491aaa9e715c886c5104eabaa1a149faa05d401c66a7f013495288e90c3cec', 'train_6_y.npy': '5f0f4a7dfacefeabdb0051e2f822406760b04d0b1187826e8a3364ecd2820978', 'train_9_y.npy': '6d945634c0405cd58df54263fa6cb65293d8dd2565c7279efa2813135b0d7f1a', 'train_18_y.npy': 'c58c0a6ab806325d705aeb079aee445618cd311751ee64e1f63c861fcc7efa4d', 'pilot.py': '3a7a6474d029b6857d2a5acf71c53a696e9e56b466efefea1307b62809a71a02', 'v7_basis.py': '7f018d9a6e388976b56f8c858aaa67e696068394623e5737049fa032220f4517', 'train_15_y.npy': 'dc08d5dba376554191ff45ea979e860f8414d0f9cf6a0d13f07ead9c5c99c5c8', 'v2_run.py': 'b5cf0daf5348bcd77dbab09876ee57950f469509f52485a15252fd873802ef2d', 'train_16_y.npy': '68c04385b7681bcaba4df93ed962187153d434d55e999e76c72ba485d9e87de0', 'rank_probe.py': '361cdeab91d0c30f2a67633cb1379dd52d9349e298e9294c3e76fe9b49545966', 'v3_math.py': '3adf452d537f36904453ed48d8f2cb0c71aed87fe274a02d8f484da23e649fe7', 'v5_anchor.py': '1cd919e1ba39424e5a1f4afeb2e7441d4215a2637f91645db14e6906b7f077ae', 'v4_run.py': 'be172c5464d0eea816a84dc2c43c012244a8c5b03e8db3b30497d2ed1354a6e9', 'v10_energy.py': 'a44777714bce1b0b6068d0d614e4f4574a9de0fbcd96420bc29bfa2e561b8284', 'train_11_y.npy': '0bb2c5c82bb953289b45c06356e42b4c55bdbd92202d999d9dab91a65f2c9ce8', 'train_3_y.npy': '0d413dbecc94f09ae50f63f694ac21bcba809345ffd2b9aba49d83813ccaaf9a', 'manifest.json': 'c67c0b5a0190e7e33c1ddc7f927de5bb0070437ea7821ce3af5c62247f64af8e', 'v4_support.py': '726777438af9ca7590cbaa6e26123d8da7bf7f05d8ee6e38cf2e45eea7b2c4f8', 'train_4_X.npy': '060a1b6436366464f249867d6979f7cbe26a9c7c5a2ecab6eaa8f4a30d99d221', 'train_5_X.npy': '1ad83b1159a69dbade233ef73ad4418aa792fcfde87c4807726b50bf7092e7e9', 'train_9_X.npy': '13ef6ca16a5a5f522d9f8ae48e4793e8450563a4b5a75cb9d120a87832082ecc', 'train_2_y.npy': 'cdfced8028548985edccbf09b37a266c5df70bad1df8865de79e3727cdbe5b04', 'train_18_X.npy': 'e7a8e89c01201e72be1fb1568b1ef74b41e7ce65f20bd1bb495964748a6b2c5f', 'train_4_y.npy': '9ce240875ae6c999c783e07440a1a9a67f1da67e13f15de7670f9bb4d53d49bc', 'train_19_X.npy': 'b802e20599277d633b2a95853d0a8ad669d901c97ecc5d080487f37eaf1351e9', 'train_1_X.npy': '2b686a2d50645fefc734902cee766a8f19e5b073028cac90752b31be0a9c6e40', 'warm_weights.pt': 'b4638d7178a6a2238d55c1036c420ab9468b4436bb20367a97f8b653a5783d2e', 'v2_methods.py': '90fa6ff5fa91cd559f9995d12a5dda0d6b9f4d6cdff435811f814adbfd664398', 'v2_math.py': '187216ea4796afb81cfc9d5350a767e8b795bcd54d24c5544a3d921be661cc62', 'teacher_base.py': 'e59ff569cd51041efe1beed91399b28f1164eaa426511c43eea2f312ce516416', 'train_3_X.npy': '221ea6e95649dbc261d202dfc31b62faf7d1308a90a53b9bbb8ad53380b8e9a6'}
MANIFEST={'v5_books.py': '1123938877061d903c6ebbb029d6f641ee7422c637181dfc93e3a0a600e7b475', 'v3_methods.py': '8c3faad01d13756bd7e27806a7fbb8d75c1f0d7c5751e54c5b4e37c66e4b0f84', 'v4_mala.py': '7d916530164cb7fa3bc9dadf051c47688327659b0e50f825e05e61ca4f805aea', 'v7_nystrom.py': '114e23dcf2ee15f6083f79fe8456edf58b08aad78d807b0ffce9b91d6dc22713', 'v9_anchor.py': '2d15e9b4f53006658a755b8d75e959eb314967773cb557cb1dbb8137ae6a0688', 'v8_run.py': 'dd362c93df255e901a274e3925a11acc9f7271ccacccd2e53c33789000dd4041', 'teacher_driver.py': 'b46da25f871c7c0fd6fdd78dde9f541168faade2f825dfe32dd86ffa4e95c968', 'v9_run.py': '2f4f460fef23e6d01315b8b80c89a2f2cf5da132119352cb50d3d15a52204614', 'v5_run.py': '9c257035c4388aaf0c48886b9e83f126bca86c812b7fe8a17843894e0200c71e', 'new_methods.py': 'b306660ea84d6bc42f3dcbbef1977e959487f1db3bd333e6b53ed3612df0d2e3', 'prepare.py': '9dd51acb3c23df4a00cec29f60f1805056d0292cfd62a809f071411550111e2e', 'archs.py': '0c24ae80e1ff5401412ea9642c9f168843c965b0dcc36942c5e85b658d9666bd', 'v6_schema.py': '46cf470ef781f5ecf9007e96be2fe0b0bd446bce49e4727fc7df19fe0ee6b654', '__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'channel.py': '609b114023990e9b7cdf30cf86b18b388e7fac2884b432ef90bc31b6399b0dc2', 'v3_dense.py': '11a1caad91455da4c013a1f48bd3ccc51c56b6cdce90ff3db26ffa9d1e40292e', 'v10_run.py': '660cc41c51514a49e8afc28f86c98ee65ab45eb45210ef93350e4f9f27902986', 'v6_run.py': 'd71baa60141a36992810f091894ff8ea84acbd99f227dc0a297e15c429a57f9f', 'dagsnet.py': '5bafe8a571934397194da42f263564ca56093547b03d7eb5735d9c0c65944a5c', 'metrics.py': '8f23006318a246673eec8c566291a5447b53a16483bcab855a9f768d1da2a712', 'v4_schema.py': '0828b4a83a75ed160643e5f0fb7c6ea418e274c8a2f9a73b01d4f3a6dc94555c', 'new_fl.py': '2d5abcb1813c1399cc29b442ed76a6eea93ce03f4600987773bb1bddec954179', 'v7_run.py': 'd2ca47dadbe17374e639569ddb277b21708031db174944842c19b9736036d428', 'model_packet.py': '3c4fc9485c936b81804865796191f34b0c353ed5acd8fc8bbe9e53b5c65b77bd', 'v6_geometry.py': 'f277b7631e109be19de998e4055a21c33a94bf2789097a2f36e2f6a2660fd2cb', 'v3_run.py': '6903adafd39211907b25385dd74ecf3063922252b364d6f172842ace588c778f', 'v8_hybrid.py': '17683b9c566310bdb22cdde361347d6af7332826c8a95589bc61d4a46b755e72', 'mobility.py': '450ae2599f4a4cbeff1a63292d492b7ab080fa8b119aea1cb7438ced7c9369d7', 'p2.py': '62889bb18a1d448f139dc5e1b01480ea444ef41bcc3ee3394568507cecd28145', 'pilot.py': '3a7a6474d029b6857d2a5acf71c53a696e9e56b466efefea1307b62809a71a02', 'v7_basis.py': '7f018d9a6e388976b56f8c858aaa67e696068394623e5737049fa032220f4517', 'v2_run.py': 'b5cf0daf5348bcd77dbab09876ee57950f469509f52485a15252fd873802ef2d', 'v3_math.py': '3adf452d537f36904453ed48d8f2cb0c71aed87fe274a02d8f484da23e649fe7', 'v5_anchor.py': '1cd919e1ba39424e5a1f4afeb2e7441d4215a2637f91645db14e6906b7f077ae', 'v4_run.py': 'be172c5464d0eea816a84dc2c43c012244a8c5b03e8db3b30497d2ed1354a6e9', 'v10_energy.py': 'a44777714bce1b0b6068d0d614e4f4574a9de0fbcd96420bc29bfa2e561b8284', 'v4_support.py': '726777438af9ca7590cbaa6e26123d8da7bf7f05d8ee6e38cf2e45eea7b2c4f8', 'v2_methods.py': '90fa6ff5fa91cd559f9995d12a5dda0d6b9f4d6cdff435811f814adbfd664398', 'v2_math.py': '187216ea4796afb81cfc9d5350a767e8b795bcd54d24c5544a3d921be661cc62', 'teacher_base.py': 'e59ff569cd51041efe1beed91399b28f1164eaa426511c43eea2f312ce516416', 'rank_math.py': '9e6b481aa31cae55053aa1a261813efdb0f2f5e2783fa34b88b9f54ad3deb3ba', 'rank_probe.py': '361cdeab91d0c30f2a67633cb1379dd52d9349e298e9294c3e76fe9b49545966', 'meta.json': '1aeb0c9156c1f7d77694ffbee6f24a3f6abeec0c0dad48c1a45741cdb85fbff4'}
PROTOCOL_SHA='34b348f27e442c5bdc146698fb11aa26ff3634bce36012f22b97d14dadb5df4a'

import hashlib,json,shutil,time
from pathlib import Path
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
import torch
assert torch.cuda.device_count()==2 and all(torch.cuda.get_device_capability(i)==(7,5) for i in range(2))
assert torch.__version__=='2.10.0+cu128'
hits=[p.parent for p in Path('/kaggle/input').rglob('protocol.json') if hashlib.sha256(p.read_bytes()).hexdigest()==PROTOCOL_SHA]
assert len(hits)==1,('input missing/ambiguous',hits)
SRC=hits[0];WORK=Path('/kaggle/temp/cross-client-rank');SOURCE=Path('/kaggle/working/source');OUT=Path('/kaggle/working/runs')
assert json.loads((SRC/'input_manifest.json').read_text())==INPUTS
for folder in (WORK/'mh',WORK/'proj',WORK/'knowledge',SOURCE,OUT):folder.mkdir(parents=True,exist_ok=True)
for name,sha in INPUTS.items():
    assert hashlib.sha256((SRC/name).read_bytes()).hexdigest()==sha,name
for name in MH:shutil.copyfile(SRC/name,WORK/'mh'/name)
for name in PROJ:shutil.copyfile(SRC/name,WORK/'proj'/name)
shutil.copyfile(SRC/'meta.json',WORK/'knowledge/meta.json')
for name in MANIFEST:shutil.copyfile(SRC/name,SOURCE/name)
for name in ('manifest.json','protocol.json','warm-proof.json','input_manifest.json'):
    shutil.copyfile(SRC/name,SOURCE/name)
(SOURCE/'source_manifest.json').write_text(json.dumps(MANIFEST,indent=2))
sys.path.insert(0,str(WORK));sys.path.insert(0,str(WORK/'proj'))
from mh.dagsnet import build_model,CFG
torch.set_num_threads(1)
warm=json.loads((SRC/'warm-proof.json').read_text())
weights=torch.load(SRC/'warm_weights.pt',map_location='cpu',weights_only=True)
assert weights['round']==2 and weights['fingerprint']==warm['fingerprint'] and len(weights['private'])==20 and not weights['proxy']
model=build_model(dict(CFG,n_features=66,num_classes=16),latent=True)
for sd in weights['private']:
    model.load_state_dict(sd,strict=True)
    assert all(torch.isfinite(v).all() for v in sd.values() if v.is_floating_point())
del model,weights
print('Exact audited CE-r2 weights/20 models/source/train data PASS',flush=True)

DIAGNOSTIC='"""Reproduce frozen probe projection; emit aggregate fit/test diagnostics, never records/gradients."""\nimport argparse\nimport json\nfrom pathlib import Path\nimport sys\nimport numpy as np\nimport torch\n\nsys.path.insert(0,str(Path(__file__).resolve().parent))\nfrom rank_probe import load_model,predict\nfrom rank_math import C,decode_up,decode_down,gradient,basis,projection\n\n\ndef measure(prob,y,logits,delta,fit):\n    design=basis(prob,y,logits);x=design[fit].reshape(-1,32);target=delta[fit].ravel()\n    scale=np.sqrt((x*x).mean(0));keep=scale>1e-12\n    coeff,_,rank,singular=np.linalg.lstsq(x[:,keep]/scale[keep],target,rcond=1e-10)\n    predicted=np.einsum(\'nkd,d->nk\',design[:,:,keep]/scale[keep],coeff)\n    residual=delta-predicted\n    frozen,_=projection(prob,y,logits,delta,fit)\n    assert np.allclose(residual[~fit],frozen,rtol=1e-10,atol=1e-12)\n    def energy(mask):\n        return dict(rows=int(mask.sum()),delta=float(np.mean(delta[mask]**2)),\n                    predicted=float(np.mean(predicted[mask]**2)),residual=float(np.mean(residual[mask]**2)))\n    classes=[]\n    for c in range(C):\n        if not np.any(y==c):classes.append(None);continue\n        item={name:energy(mask&(y==c)) for name,mask in [(\'fit\',fit),(\'test\',~fit)]}\n        e=np.sort((residual[(~fit)&(y==c)]**2).sum(1))[::-1]\n        item[\'test_residual_top1_fraction\']=float(e[0]/e.sum()) if e.sum()>0 else 0.\n        item[\'test_residual_top5_fraction\']=float(e[:5].sum()/e.sum()) if e.sum()>0 else 0.\n        classes.append(item)\n    original=np.zeros(32);original[keep]=coeff/scale[keep]\n    return dict(rank=int(rank),columns=int(keep.sum()),fit_scale=scale.tolist(),\n        coefficients_normalized=coeff.tolist(),coefficients_original=original.tolist(),\n        singular_values=singular.tolist(),condition=float(singular[0]/singular[-1]) if singular[-1]>0 else None,\n        fit=energy(fit),test=energy(~fit),classes=classes)\n\n\ndef main():\n    p=argparse.ArgumentParser();p.add_argument(\'--input\',type=Path,required=True);p.add_argument(\'--out\',type=Path,required=True)\n    p.add_argument(\'--seed\',type=int,required=True);p.add_argument(\'--shard\',type=int,required=True);a=p.parse_args()\n    import hashlib\n    identity=hashlib.sha256((a.input/\'protocol.json\').read_bytes()).digest()[:16]\n    replay=json.loads((a.out/\'replay.json\').read_text());pooled,ng,eligible,total=decode_down(bytes.fromhex(replay[\'cohort\']),identity)\n    assert torch.cuda.device_count()==1\n    torch.set_num_threads(1);torch.backends.cuda.matmul.allow_tf32=False\n    torch.use_deterministic_algorithms(True);torch.backends.cudnn.benchmark=False;torch.backends.cudnn.deterministic=True\n    weights=torch.load(a.input/\'warm_weights.pt\',map_location=\'cpu\',weights_only=True)\n    assert weights[\'round\']==2 and len(weights[\'private\'])==20 and not weights[\'proxy\']\n    for k in range(a.shard,20,2):\n        n,s,_=decode_up(bytes.fromhex(replay[\'up\'][k]),identity);own=np.divide(s[:2],n,out=np.zeros((2,C)),where=n>0)\n        support=eligible&(n>=32).all(0);x=np.load(a.input/f\'train_{k}_X.npy\',mmap_mode=\'r\');y=np.load(a.input/f\'train_{k}_y.npy\')\n        indexes=[];fit=[]\n        for c in np.flatnonzero(support):\n            available=np.flatnonzero(y==c);rng=np.random.default_rng(np.random.SeedSequence([1729,k,int(c)]))\n            chosen=rng.choice(available,min(256,len(available)),replace=False);indexes.extend(chosen.tolist())\n            fit.extend([True]*(len(chosen)//2)+[False]*(len(chosen)-len(chosen)//2))\n        assert indexes,\'original probe had no common support\'\n        indexes=np.asarray(indexes);fit=np.asarray(fit,dtype=bool);yy=y[indexes]\n        model=load_model(weights,k,\'cuda:0\');z,prob=predict(model,x[indexes],\'cuda:0\')\n        go=gradient(prob,yy,own,n,int(n[0].sum()),support);delta=gradient(prob,yy,pooled,ng,total,support)-go\n        diag=measure(prob,yy,z.astype(\'float64\'),delta,fit)\n        for c,item in enumerate(diag[\'classes\']):\n            reference=replay[\'clients\'][k][\'classes\'][c]\n            assert (item is None)==(reference is None)\n            if item is not None:\n                selected=(~fit)&(yy==c)\n                for name,value in [(\'own_energy\',float(np.mean(go[selected]**2))),(\'delta_energy\',item[\'test\'][\'delta\']),(\'residual_energy\',item[\'test\'][\'residual\'])]:\n                    assert np.isclose(value,reference[name],rtol=1e-6,atol=1e-18),(\'original energy mismatch\',k,c,name,value,reference[name])\n        diag.update(client=k,seed=a.seed,original_energies_reproduced=True)\n        (a.out/f\'diagnostic_{k:02d}.json\').write_text(json.dumps(diag,indent=2,allow_nan=False)+\'\\n\')\n        print(\'DIAGNOSTIC\',a.seed,k,\'condition\',diag[\'condition\'],\'rank\',diag[\'rank\'],flush=True)\n        del model;torch.cuda.empty_cache()\n\n\nif __name__==\'__main__\':main()\n'
DIAGNOSTIC_SHA='51a45e101f2e614203a826e858cc27d8b6c1758e3d4eeb5c1b68ab7519a7153d'
REPLAY={'parent': 'minhtriethihi/mobihet-ccr-probe-s42-20261015-v1', 'audit_sha': '3ec6f566aab2e8173e9d088953e12ae87f3949df794a09488286c8bc3e5eec0a', 'cohort': '4d48435201020200ca020000e072afb634b348f27e442c5bdc146698fb11aa268bb0df1aa9adea3fbd2038b5559ee93fe53bf8151421ea3f618c90d45ca2e43f113081c15975e43f62b146e66eb2ef3ff1c0da963ec2e83fe2aabf96135fe23f2b4bf3f52c32ee3f7bae404575ebeb3fbe613bbc4879eb3f26bca3618184de3f8ce296ef4993ea3f44f3c2246cd5e23fa659c4934c36ef3fda94b1f402aaeb3f1b9716ee8f00a63f7632d500eee3623fd8e8494ad0f67f3f03acc3904c8c923f6066ed03d4aa933f85642a23ce475a3f7f73c148c9c9623fa73e3e52e0fd913fe671116ecb95673f0050e268c390773ffad728beacc1723f026cad59a65d6b3f1916fbf853d56f3fae520dd76703943f67ae3531dea57e3f967acca27ac16c3fdf58030000000000c43d00000000000004a20000000000007b9e0000000000005cad000000000000ca34020000000000092f00000000000067a900000000000025ac0000000000009dc500000000000038640000000000005f190000000000000e4f000000000000c6b1000000000000b05d0300000000001c64000000000000d2ea0b0000000000ed050f0000000000ada10e000000000036a50e000000000055960e0000000000e70e0d0000000000a8140f00000000004a9a0e00000000008c970e0000000000147e0e000000000079df0e0000000000522a0f0000000000a3f40e0000000000eb910e000000000001e60b000000000095df0e00000000001400000014000000140000001400000014000000140000001400000014000000140000001400000014000000140000001400000014000000140000001400000014000000140000001400000014000000140000001400000014000000140000001400000014000000140000001400000014000000140000001400000014000000ffffc789603e817d813c0dd3293d741e263df1b6353d0700143e5f38453c2591313df670343d0b234f3dec19d23c6dc7d43bedbba53c60573a3de0cc613e39dfd13cb1430f0000000000', 'up': ['...'], 'clients': [{'client': 0, 'classes': [{'rows': 128, 'own_energy': 4.304008288550379e-06, 'delta_energy': 4.665554183110264e-07, 'residual_energy': 2.162849551249502e-05}, {'rows': 128, 'own_energy': 0.04007230099316857, 'delta_energy': 0.022467545061353685, 'residual_energy': 0.00857995352997274}, {'rows': 128, 'own_energy': 0.015305212198071495, 'delta_energy': 0.011054993370545791, 'residual_energy': 0.005478273444509674}, {'rows': 35, 'own_energy': 0.49375685949545406, 'delta_energy': 0.47147322799599994, 'residual_energy': 0.21386412705219274}, {'rows': 128, 'own_energy': 0.019981136619039122, 'delta_energy': 0.015166757519297164, 'residual_energy': 0.0065321006822379145}, {'rows': 128, 'own_energy': 3.2867738206669964e-07, 'delta_energy': 6.495771697701748e-08, 'residual_energy': 4.9259319637159436e-05}, {'rows': 128, 'own_energy': 0.0023188113689574445, 'delta_energy': 0.0001149614191105492, 'residual_energy': 5.688560457227306e-05}, {'rows': 128, 'own_energy': 0.000362312120480218, 'delta_energy': 0.00011265027698594354, 'residual_energy': 3.6582722420122384e-05}, {'rows': 128, 'own_energy': 5.767360569537701e-05, 'delta_energy': 2.6705299366457197e-06, 'residual_energy': 3.483464779268992e-05}, {'rows': 128, 'own_energy': 3.38651582824632e-05, 'delta_energy': 1.8998547206406086e-06, 'residual_energy': 2.1777872623481922e-05}, {'rows': 128, 'own_energy': 0.00024940384283986196, 'delta_energy': 2.1622115918677425e-05, 'residual_energy': 2.5330287050694022e-05}, {'rows': 128, 'own_energy': 0.011913656577703682, 'delta_energy': 0.0004071914230564054, 'residual_energy': 0.00015450220206331936}, {'rows': 38, 'own_energy': 0.8297056505392916, 'delta_energy': 0.7514724879711694, 'residual_energy': 0.4073890981012774}, {'rows': 128, 'own_energy': 0.014356361928832924, 'delta_energy': 0.0108166190623629, 'residual_energy': 0.004584266705574838}, {'rows': 128, 'own_energy': 1.0571590584346562e-06, 'delta_energy': 3.5900620975719968e-09, 'residual_energy': 3.051768980313918e-05}, {'rows': 128, 'own_energy': 0.006086981032520624, 'delta_energy': 0.0030104679130073617, 'residual_energy': 0.0011004457081800316}], 'projection': {'columns': 32, 'retained_columns': 32, 'fit_rows': 1863, 'test_rows': 1865}, 'scan_rows': 78372, 'valid_scan_rows': 78372, 'support': [True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True]}, {'client': 1, 'classes': [{'rows': 32, 'own_energy': 0.33064682797609873, 'delta_energy': 0.3251284168357786, 'residual_energy': 0.05443962002007545}, {'rows': 128, 'own_energy': 0.00016059660886921188, 'delta_energy': 0.00013769724198645607, 'residual_energy': 4.127955203387912e-05}, {'rows': 128, 'own_energy': 1.8335991166859345e-05, 'delta_energy': 5.983654081696704e-05, 'residual_energy': 1.630816441893194e-05}, {'rows': 49, 'own_energy': 0.0930478165064254, 'delta_energy': 0.0818673625412745, 'residual_energy': 0.020384945700009507}, {'rows': 128, 'own_energy': 9.075249988453985e-05, 'delta_energy': 9.397699771503575e-05, 'residual_energy': 1.6066439095736874e-05}, {'rows': 128, 'own_energy': 1.0367229159627833e-07, 'delta_energy': 1.3292913892896913e-07, 'residual_energy': 9.98162008868887e-06}, {'rows': 128, 'own_energy': 0.0005031790876636006, 'delta_energy': 0.0004358413813395491, 'residual_energy': 0.00016420826887604458}, {'rows': 128, 'own_energy': 0.0002924183077997263, 'delta_energy': 5.349742438381426e-05, 'residual_energy': 1.4083650324301262e-05}, {'rows': 128, 'own_energy': 1.707589928321297e-06, 'delta_energy': 5.097143290459039e-06, 'residual_energy': 2.0769373859301356e-05}, {'rows': 59, 'own_energy': 0.08740216823568543, 'delta_energy': 0.07720115981871332, 'residual_energy': 0.0377453023770127}, {'rows': 128, 'own_energy': 1.384154456130089e-05, 'delta_energy': 4.640342097313585e-05, 'residual_energy': 2.141148618120192e-05}, {'rows': 128, 'own_energy': 0.005902339406115278, 'delta_energy': 0.0025165557938484143, 'residual_energy': 0.0007864592519197512}, {'rows': 85, 'own_energy': 0.02835561466515784, 'delta_energy': 0.01765548679231338, 'residual_energy': 0.002117327153192181}, {'rows': 128, 'own_energy': 0.007205407048873623, 'delta_energy': 0.00345562843257231, 'residual_energy': 0.0006488821408378164}, {'rows': 92, 'own_energy': 0.02954988310790221, 'delta_energy': 0.028164812760580114, 'residual_energy': 0.0033866640616971792}, {'rows': 59, 'own_energy': 0.025006728986499666, 'delta_energy': 0.019497077403651062, 'residual_energy': 0.0061668895926720296}], 'projection': {'columns': 32, 'retained_columns': 32, 'fit_rows': 1654, 'test_rows': 1656}, 'scan_rows': 40403, 'valid_scan_rows': 40403, 'support': [True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True]}, {'client': 2, 'classes': [{'rows': 128, 'own_energy': 1.7904150514670115e-06, 'delta_energy': 3.4663121877002023e-07, 'residual_energy': 2.144447348458643e-05}, {'rows': 128, 'own_energy': 0.0005882903439159671, 'delta_energy': 1.5983820063801424e-06, 'residual_energy': 3.242167047946053e-05}, {'rows': 128, 'own_energy': 0.0042759670844666, 'delta_energy': 0.00261042443318099, 'residual_energy': 0.0007874402697797043}, {'rows': 34, 'own_energy': 0.8244516700960706, 'delta_energy': 0.7784195376639182, 'residual_energy': 0.1455046512643501}, {'rows': 72, 'own_energy': 0.07773314448561915, 'delta_energy': 0.06922813270067882, 'residual_energy': 0.019458446129710462}, {'rows': 128, 'own_energy': 2.537235478383886e-07, 'delta_energy': 1.2232414388778064e-08, 'residual_energy': 6.878897585926794e-05}, {'rows': 128, 'own_energy': 0.005400231322412661, 'delta_energy': 0.0009032318469294569, 'residual_energy': 0.0003352074405831835}, {'rows': 128, 'own_energy': 0.001520679655924873, 'delta_energy': 5.451335658680104e-05, 'residual_energy': 3.305416249081211e-05}, {'rows': 128, 'own_energy': 5.684546983490973e-06, 'delta_energy': 7.823164132938035e-06, 'residual_energy': 5.2558764491597205e-05}, {'rows': 128, 'own_energy': 0.0006063774357748528, 'delta_energy': 0.00022601290288770985, 'residual_energy': 8.547260407215369e-05}, {'rows': 128, 'own_energy': 0.0019454883085166903, 'delta_energy': 0.0006867100614542732, 'residual_energy': 0.0004110751606156749}, {'rows': 22, 'own_energy': 1.7623671516339485, 'delta_energy': 1.393466414383045, 'residual_energy': 0.6448449608319398}, {'rows': 37, 'own_energy': 0.5317319447482521, 'delta_energy': 0.4690067525967028, 'residual_energy': 0.13527123070007113}, {'rows': 65, 'own_energy': 0.0882284879255301, 'delta_energy': 0.07967844443386875, 'residual_energy': 0.020113034293778437}, {'rows': 128, 'own_energy': 0.0001381424466239297, 'delta_energy': 9.97900319390503e-05, 'residual_energy': 4.1123444799022354e-05}, {'rows': 128, 'own_energy': 1.9244254414454104e-05, 'delta_energy': 3.111515336373053e-05, 'residual_energy': 5.4839267063544085e-05}], 'projection': {'columns': 32, 'retained_columns': 32, 'fit_rows': 1637, 'test_rows': 1638}, 'scan_rows': 61618, 'valid_scan_rows': 61618, 'support': [True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True]}, {'client': 3, 'classes': [{'rows': 128, 'own_energy': 5.3338484547443486e-05, 'delta_energy': 2.9249019005009533e-06, 'residual_energy': 3.278843799559756e-06}, {'rows': 128, 'own_energy': 0.0007149005009345324, 'delta_energy': 0.00012591101755473453, 'residual_energy': 4.100223804074159e-05}, {'rows': 128, 'own_energy': 0.00010167932967641513, 'delta_energy': 2.8678456760302157e-05, 'residual_energy': 1.4601832665542925e-05}, {'rows': 128, 'own_energy': 0.0002572120451435374, 'delta_energy': 0.00010469474467481499, 'residual_energy': 1.9366618245181853e-05}, {'rows': 128, 'own_energy': 9.538575429927373e-06, 'delta_energy': 8.545870768648605e-05, 'residual_energy': 2.209508120008129e-05}, {'rows': 128, 'own_energy': 8.969280347911444e-08, 'delta_energy': 5.303795912413592e-09, 'residual_energy': 3.532714645043855e-06}, {'rows': 128, 'own_energy': 0.00011117440841235322, 'delta_energy': 0.000893058256704269, 'residual_energy': 0.0002876777541378502}, {'rows': 57, 'own_energy': 0.02695007517376112, 'delta_energy': 0.022723916836823413, 'residual_energy': 0.008735423227199503}, {'rows': 128, 'own_energy': 1.0698489212559741e-05, 'delta_energy': 5.525352319792353e-06, 'residual_energy': 5.885297042625494e-06}, {'rows': 60, 'own_energy': 0.06078922700913308, 'delta_energy': 0.05221689913424421, 'residual_energy': 0.02377449913403525}, {'rows': 128, 'own_energy': 0.0004313636332215554, 'delta_energy': 5.862363709779067e-06, 'residual_energy': 5.740672498559283e-06}, {'rows': 69, 'own_energy': 0.05503025060244445, 'delta_energy': 0.00788568352314898, 'residual_energy': 0.0028271513279598884}, {'rows': 128, 'own_energy': 0.0002881555505907505, 'delta_energy': 3.0679521224923474e-05, 'residual_energy': 9.92994845875089e-06}, {'rows': 128, 'own_energy': 0.0034639442494868826, 'delta_energy': 0.0016658042753121892, 'residual_energy': 0.000464280171535973}, {'rows': 128, 'own_energy': 2.4435957182855962e-05, 'delta_energy': 1.2083928663868445e-05, 'residual_energy': 4.878172288294203e-06}, {'rows': 128, 'own_energy': 0.0003266663038511246, 'delta_energy': 1.1963537188496757e-05, 'residual_energy': 7.864982674975447e-06}], 'projection': {'columns': 32, 'retained_columns': 32, 'fit_rows': 1849, 'test_rows': 1850}, 'scan_rows': 33858, 'valid_scan_rows': 33858, 'support': [True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True]}, {'client': 4, 'classes': [{'rows': 128, 'own_energy': 2.7993935001685588e-05, 'delta_energy': 6.069392569940035e-08, 'residual_energy': 4.11811053931086e-06}, {'rows': 128, 'own_energy': 0.0020976142818654253, 'delta_energy': 4.795132967295829e-05, 'residual_energy': 2.26486378225861e-05}, {'rows': 128, 'own_energy': 3.268239918102714e-05, 'delta_energy': 6.661742717301033e-05, 'residual_energy': 3.5772434091182266e-05}, {'rows': 128, 'own_energy': 0.0001478335619050128, 'delta_energy': 5.8498703213229964e-05, 'residual_energy': 1.4252135488487491e-05}, {'rows': 128, 'own_energy': 0.0013789850229920983, 'delta_energy': 0.0003326470465375145, 'residual_energy': 0.00012855754533439134}, {'rows': 128, 'own_energy': 4.103817326245639e-07, 'delta_energy': 3.8699965101425884e-08, 'residual_energy': 3.248311265894644e-05}, {'rows': 128, 'own_energy': 0.00024869250650628864, 'delta_energy': 0.0005713221671702653, 'residual_energy': 9.074236401030702e-05}, {'rows': 96, 'own_energy': 0.022552436718829127, 'delta_energy': 0.017993473186542642, 'residual_energy': 0.006083331271165389}, {'rows': 128, 'own_energy': 0.0009607207231963316, 'delta_energy': 0.0003990170561747199, 'residual_energy': 0.00016452541240016567}, {'rows': 34, 'own_energy': 0.20573356725870026, 'delta_energy': 0.19288139844034988, 'residual_energy': 0.08131058867222792}, {'rows': 128, 'own_energy': 0.0037915163279564307, 'delta_energy': 0.0014286833487392402, 'residual_energy': 0.000600802341537807}, {'rows': 128, 'own_energy': 0.00022445269155652901, 'delta_energy': 0.0035995163898715626, 'residual_energy': 0.0015177982387805393}, {'rows': 128, 'own_energy': 0.0004633082804248446, 'delta_energy': 0.0001241105156684429, 'residual_energy': 4.155624222908123e-05}, {'rows': 128, 'own_energy': 6.185014418870741e-05, 'delta_energy': 0.00015399123887665606, 'residual_energy': 2.6421658463987166e-05}, {'rows': 128, 'own_energy': 4.249795588666088e-06, 'delta_energy': 2.9343784106249617e-07, 'residual_energy': 6.5834869442578095e-06}, {'rows': 73, 'own_energy': 0.09270148906871503, 'delta_energy': 0.06971134699333731, 'residual_energy': 0.017606690036214956}], 'projection': {'columns': 32, 'retained_columns': 32, 'fit_rows': 1867, 'test_rows': 1867}, 'scan_rows': 44034, 'valid_scan_rows': 44034, 'support': [True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True]}, {'client': 5, 'classes': [{'rows': 128, 'own_energy': 5.592555711047333e-06, 'delta_energy': 2.325887641841676e-06, 'residual_energy': 9.228721084164215e-06}, {'rows': 128, 'own_energy': 0.00042707859104205206, 'delta_energy': 0.00013792822142068116, 'residual_energy': 5.095036983135168e-05}, {'rows': 107, 'own_energy': 0.0226520663603908, 'delta_energy': 0.01757603421025712, 'residual_energy': 0.0049667194799392885}, {'rows': 128, 'own_energy': 0.009221324938410276, 'delta_energy': 0.004952614143740644, 'residual_energy': 0.0012215680500832354}, {'rows': 128, 'own_energy': 0.00010746759753340827, 'delta_energy': 6.1029300863441834e-05, 'residual_energy': 1.7258133238724795e-05}, {'rows': 128, 'own_energy': 5.633791594135083e-06, 'delta_energy': 1.3854669320181405e-06, 'residual_energy': 1.2767504053770871e-05}, {'rows': 105, 'own_energy': 0.0208778129912513, 'delta_energy': 0.007778524621397278, 'residual_energy': 0.0023498461815236868}, {'rows': 128, 'own_energy': 0.001762916924495036, 'delta_energy': 0.0001925819116569937, 'residual_energy': 4.4939895512167355e-05}, {'rows': 128, 'own_energy': 0.00014215704871828658, 'delta_energy': 1.7562087901861123e-05, 'residual_energy': 2.1465716214366587e-05}, {'rows': 74, 'own_energy': 0.06730775103404264, 'delta_energy': 0.05850263657197176, 'residual_energy': 0.021170847543051576}, {'rows': 128, 'own_energy': 0.00011108723271161826, 'delta_energy': 2.573114664459517e-05, 'residual_energy': 1.6141761753894128e-05}, {'rows': 40, 'own_energy': 0.13162976930195577, 'delta_energy': 0.06909985985682794, 'residual_energy': 0.030229909557405944}, {'rows': 128, 'own_energy': 0.0017292771210936316, 'delta_energy': 6.89409879884476e-05, 'residual_energy': 2.4827636418483185e-05}, {'rows': 34, 'own_energy': 0.26324382846947336, 'delta_energy': 0.24546267966731286, 'residual_energy': 0.06098909153889947}, {'rows': 128, 'own_energy': 8.720185988298889e-06, 'delta_energy': 1.855672395740769e-06, 'residual_energy': 8.95818381511425e-06}, {'rows': 128, 'own_energy': 0.00023196141947186107, 'delta_energy': 1.2511373709202606e-06, 'residual_energy': 7.396743916319371e-06}], 'projection': {'columns': 32, 'retained_columns': 32, 'fit_rows': 1766, 'test_rows': 1768}, 'scan_rows': 44969, 'valid_scan_rows': 44969, 'support': [True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True]}, {'client': 6, 'classes': [{'rows': 128, 'own_energy': 3.463726029125143e-06, 'delta_energy': 4.732676143622988e-07, 'residual_energy': 7.941504853797924e-05}, {'rows': 128, 'own_energy': 0.001406552153749967, 'delta_energy': 0.00012627848781775254, 'residual_energy': 0.00011901401780785305}, {'rows': 128, 'own_energy': 0.0053565502519160885, 'delta_energy': 0.003044443477767996, 'residual_energy': 0.0014414058753892062}, {'rows': 128, 'own_energy': 2.1447965089478876e-05, 'delta_energy': 6.110089466756624e-05, 'residual_energy': 6.072882891478468e-05}, {'rows': 128, 'own_energy': 0.004285307082907975, 'delta_energy': 0.0019471716193640146, 'residual_energy': 0.0007696566919043807}, {'rows': 128, 'own_energy': 9.154506792583251e-10, 'delta_energy': 3.489610155909751e-10, 'residual_energy': 9.49964403185145e-05}, {'rows': 128, 'own_energy': 0.0344840803359466, 'delta_energy': 0.016482349290739017, 'residual_energy': 0.004960455796796281}, {'rows': 128, 'own_energy': 0.0023951627878550192, 'delta_energy': 0.0004769051398252807, 'residual_energy': 0.00016727105563617257}, {'rows': 128, 'own_energy': 2.295355117077396e-06, 'delta_energy': 3.123998300073544e-06, 'residual_energy': 0.00011112367947606206}, {'rows': 128, 'own_energy': 0.007470389927270763, 'delta_energy': 0.005381116892499727, 'residual_energy': 0.0024912978032017976}, {'rows': 106, 'own_energy': 0.10266604667155872, 'delta_energy': 0.08179448607422114, 'residual_energy': 0.0275268493481243}, {'rows': 95, 'own_energy': 0.132861398699599, 'delta_energy': 0.05223662149841162, 'residual_energy': 0.02458049301388172}, {'rows': 74, 'own_energy': 0.2169377831902126, 'delta_energy': 0.17769701590262652, 'residual_energy': 0.07055751719784269}, {'rows': 128, 'own_energy': 0.03242607631880007, 'delta_energy': 0.026563935843925707, 'residual_energy': 0.009835159005545243}, {'rows': 128, 'own_energy': 1.9315596676918082e-07, 'delta_energy': 5.416892698399293e-09, 'residual_energy': 8.307450853703385e-05}, {'rows': 128, 'own_energy': 0.012781670986447558, 'delta_energy': 0.008017312281212788, 'residual_energy': 0.0028635122794367735}], 'projection': {'columns': 32, 'retained_columns': 32, 'fit_rows': 1936, 'test_rows': 1939}, 'scan_rows': 78544, 'valid_scan_rows': 78544, 'support': [True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True]}, {'client': 7, 'classes': [{'rows': 128, 'own_energy': 5.5650164861143276e-05, 'delta_energy': 6.270909591361251e-06, 'residual_energy': 2.919938044652177e-06}, {'rows': 128, 'own_energy': 0.0010977294898349623, 'delta_energy': 9.624782761469256e-06, 'residual_energy': 8.979604513798684e-06}, {'rows': 128, 'own_energy': 0.00031838892387398503, 'delta_energy': 1.153957140572506e-05, 'residual_energy': 1.0727446138824662e-05}, {'rows': 128, 'own_energy': 0.0002031440008750183, 'delta_energy': 9.589239088937296e-05, 'residual_energy': 2.1189546768084974e-05}, {'rows': 128, 'own_energy': 0.00132454259024784, 'delta_energy': 0.00024099121362550717, 'residual_energy': 7.47915290106174e-05}, {'rows': 128, 'own_energy': 0.00045129581675807794, 'delta_energy': 0.00037660201809607365, 'residual_energy': 0.00026447349901302555}, {'rows': 128, 'own_energy': 8.897397211644908e-05, 'delta_energy': 0.0006930966014542628, 'residual_energy': 0.00037689235710471286}, {'rows': 128, 'own_energy': 4.2913039232032676e-05, 'delta_energy': 0.0001783602734746897, 'residual_energy': 2.591516599354311e-05}, {'rows': 40, 'own_energy': 0.10352474310075362, 'delta_energy': 0.09240465457187375, 'residual_energy': 0.04555104568631761}, {'rows': 128, 'own_energy': 0.00015429036252394683, 'delta_energy': 1.4164867415935283e-06, 'residual_energy': 5.5841712674821505e-06}, {'rows': 33, 'own_energy': 0.22139328489517754, 'delta_energy': 0.1889955320711285, 'residual_energy': 0.06869210935036135}, {'rows': 62, 'own_energy': 0.027038949704538785, 'delta_energy': 0.005376692121989285, 'residual_energy': 0.0029747806185542844}, {'rows': 128, 'own_energy': 0.00487326796994983, 'delta_energy': 0.0004992122298521963, 'residual_energy': 0.0001825324997955031}, {'rows': 128, 'own_energy': 0.0008449778153739444, 'delta_energy': 0.0001589044975990211, 'residual_energy': 3.200497155152029e-05}, {'rows': 128, 'own_energy': 3.067198814837319e-07, 'delta_energy': 1.3941832670980709e-08, 'residual_energy': 6.798069125425506e-06}, {'rows': 128, 'own_energy': 0.0018761165338827752, 'delta_energy': 0.00027487853922757334, 'residual_energy': 8.183165653843726e-05}], 'projection': {'columns': 32, 'retained_columns': 32, 'fit_rows': 1799, 'test_rows': 1799}, 'scan_rows': 34525, 'valid_scan_rows': 34525, 'support': [True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True]}, {'client': 8, 'classes': [{'rows': 128, 'own_energy': 1.3394397887066349e-06, 'delta_energy': 6.259584555419381e-07, 'residual_energy': 6.890938396517428e-06}, {'rows': 128, 'own_energy': 0.0027306261973698838, 'delta_energy': 0.0002357040278996787, 'residual_energy': 8.52969241257383e-05}, {'rows': 128, 'own_energy': 0.0007410945942708156, 'delta_energy': 0.00016244378863175, 'residual_energy': 9.643799138657288e-05}, {'rows': 128, 'own_energy': 0.000142117639341101, 'delta_energy': 1.8768627421412216e-05, 'residual_energy': 1.5102993528697161e-05}, {'rows': 128, 'own_energy': 0.004735159552895967, 'delta_energy': 0.002529920780792957, 'residual_energy': 0.0010315037126607577}, {'rows': 128, 'own_energy': 4.5018770587465225e-05, 'delta_energy': 2.554504931461368e-05, 'residual_energy': 2.389211543238121e-05}, {'rows': 128, 'own_energy': 0.0019874955948925325, 'delta_energy': 4.42905815501325e-06, 'residual_energy': 8.19729196863009e-06}, {'rows': 128, 'own_energy': 0.015545406401957345, 'delta_energy': 0.01156574376425735, 'residual_energy': 0.00431689044190171}, {'rows': 73, 'own_energy': 0.056414248701480804, 'delta_energy': 0.050555332228499145, 'residual_energy': 0.018021448684207117}, {'rows': 128, 'own_energy': 0.001887062997569448, 'delta_energy': 0.0010701072103376716, 'residual_energy': 0.0004926591693868362}, {'rows': 128, 'own_energy': 4.5761376666073255e-05, 'delta_energy': 2.853229608372366e-05, 'residual_energy': 2.8383979341094835e-05}, {'rows': 68, 'own_energy': 0.19373696087667377, 'delta_energy': 0.08984919217676435, 'residual_energy': 0.044554735629148794}, {'rows': 128, 'own_energy': 0.0002209603226992668, 'delta_energy': 3.1950688776971674e-05, 'residual_energy': 2.742979847197799e-05}, {'rows': 128, 'own_energy': 0.0025343631129212613, 'delta_energy': 0.0007089752788644576, 'residual_energy': 0.00020678581607774625}, {'rows': 128, 'own_energy': 3.279030159383861e-05, 'delta_energy': 1.6938532161245127e-05, 'residual_energy': 1.090052819121846e-05}, {'rows': 128, 'own_energy': 0.0004338639514505801, 'delta_energy': 2.2512458330764065e-05, 'residual_energy': 1.995486550270263e-05}], 'projection': {'columns': 32, 'retained_columns': 32, 'fit_rows': 1932, 'test_rows': 1933}, 'scan_rows': 65629, 'valid_scan_rows': 65629, 'support': [True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True]}, {'client': 9, 'classes': [{'rows': 128, 'own_energy': 0.00044596324968979457, 'delta_energy': 0.00022735353114154697, 'residual_energy': 6.23242918726856e-05}, {'rows': 33, 'own_energy': 0.0840350373567677, 'delta_energy': 0.06178642963528053, 'residual_energy': 0.018458028000834268}, {'rows': 44, 'own_energy': 0.059471907966079644, 'delta_energy': 0.05094559239367924, 'residual_energy': 0.023368976132218455}, {'rows': 128, 'own_energy': 0.00017995943816573678, 'delta_energy': 0.00020155915153880258, 'residual_energy': 3.06629880265715e-05}, {'rows': 128, 'own_energy': 0.001674977389084874, 'delta_energy': 0.0004420543061928751, 'residual_energy': 0.0001185039265863327}, {'rows': 128, 'own_energy': 3.46866511481694e-09, 'delta_energy': 6.209782707990295e-10, 'residual_energy': 7.514242841665394e-06}, {'rows': 65, 'own_energy': 0.04127629743953348, 'delta_energy': 0.0162664790327032, 'residual_energy': 0.0029793642806023083}, {'rows': 128, 'own_energy': 0.004459916614068662, 'delta_energy': 0.001831908204419056, 'residual_energy': 0.0003570336488182642}, {'rows': 128, 'own_energy': 0.0010649485619271958, 'delta_energy': 0.0005430923125678166, 'residual_energy': 0.0002163274865062751}, {'rows': 128, 'own_energy': 2.0537328007465485e-06, 'delta_energy': 1.984520065871329e-05, 'residual_energy': 1.2698112052426726e-05}, {'rows': 128, 'own_energy': 0.00022485358860749923, 'delta_energy': 4.650368641581821e-06, 'residual_energy': 5.148524841206625e-06}, {'rows': 41, 'own_energy': 0.026177374131875385, 'delta_energy': 0.008331025252527815, 'residual_energy': 0.0035261461215615837}, {'rows': 128, 'own_energy': 0.006575973277530529, 'delta_energy': 0.001643774929282204, 'residual_energy': 0.0005463608598892487}, {'rows': 128, 'own_energy': 2.513433823849314e-06, 'delta_energy': 3.590503292823207e-05, 'residual_energy': 1.1324456821845822e-05}, {'rows': 16, 'own_energy': 0.8646625904827397, 'delta_energy': 0.8549575668561035, 'residual_energy': 0.03823799055741795}, {'rows': 128, 'own_energy': 4.691222495621426e-05, 'delta_energy': 2.253609485070318e-05, 'residual_energy': 8.041113579612042e-06}], 'projection': {'columns': 32, 'retained_columns': 32, 'fit_rows': 1605, 'test_rows': 1607}, 'scan_rows': 29080, 'valid_scan_rows': 29080, 'support': [True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True]}, {'client': 10, 'classes': [{'rows': 128, 'own_energy': 0.0063377161709912665, 'delta_energy': 0.005345545787864365, 'residual_energy': 0.0007665652078461094}, {'rows': 128, 'own_energy': 1.79693521394835e-05, 'delta_energy': 0.0001241031457499248, 'residual_energy': 8.256669424128562e-05}, {'rows': 128, 'own_energy': 0.0003048420738350813, 'delta_energy': 1.3490171346663591e-05, 'residual_energy': 7.989893233358704e-06}, {'rows': 128, 'own_energy': 0.0005758474161967544, 'delta_energy': 3.0598055971758065e-05, 'residual_energy': 7.134228130903985e-06}, {'rows': 128, 'own_energy': 7.429415276497794e-05, 'delta_energy': 6.024372698339677e-05, 'residual_energy': 2.0889614722746386e-05}, {'rows': 128, 'own_energy': 3.899810433647903e-10, 'delta_energy': 4.45261728222223e-11, 'residual_energy': 4.207226948687586e-06}, {'rows': 91, 'own_energy': 0.07219524175520305, 'delta_energy': 0.04111786850142762, 'residual_energy': 0.01114924904865236}, {'rows': 128, 'own_energy': 0.0005227667280451724, 'delta_energy': 3.143681199886572e-05, 'residual_energy': 5.793021392244555e-06}, {'rows': 128, 'own_energy': 0.0021502197985330037, 'delta_energy': 0.0011274647767077041, 'residual_energy': 0.00044223715124203767}, {'rows': 128, 'own_energy': 0.0007140619761419873, 'delta_energy': 0.0002799015314551501, 'residual_energy': 0.00012980112901373852}, {'rows': 128, 'own_energy': 0.0008882593179552053, 'delta_energy': 0.000124442253534281, 'residual_energy': 5.929189702078983e-05}, {'rows': 128, 'own_energy': 0.004344529496554839, 'delta_energy': 0.00903076683819828, 'residual_energy': 0.0023147842925729566}, None, {'rows': 128, 'own_energy': 0.00014074314254446126, 'delta_energy': 7.426142337329809e-05, 'residual_energy': 6.609524694477132e-06}, {'rows': 128, 'own_energy': 2.6615482111604267e-06, 'delta_energy': 1.3533053794149808e-07, 'residual_energy': 3.588462907792212e-06}, {'rows': 128, 'own_energy': 3.0221824377509684e-05, 'delta_energy': 0.00011033743337536508, 'residual_energy': 1.8979373297736604e-05}], 'projection': {'columns': 32, 'retained_columns': 31, 'fit_rows': 1882, 'test_rows': 1883}, 'scan_rows': 62103, 'valid_scan_rows': 62103, 'support': [True, True, True, True, True, True, True, True, True, True, True, True, False, True, True, True]}, {'client': 11, 'classes': [{'rows': 128, 'own_energy': 7.592163374342654e-06, 'delta_energy': 4.720089771271462e-08, 'residual_energy': 1.090584186713627e-05}, {'rows': 56, 'own_energy': 0.1172985414944595, 'delta_energy': 0.08141691015435415, 'residual_energy': 0.025152227801266608}, {'rows': 128, 'own_energy': 0.0023484459566444297, 'delta_energy': 0.0011715546802437793, 'residual_energy': 0.0008242205042115324}, {'rows': 25, 'own_energy': 0.494630058343264, 'delta_energy': 0.46463743919839856, 'residual_energy': 0.10383275005977165}, {'rows': 128, 'own_energy': 4.961529228748521e-05, 'delta_energy': 0.00011921730164145223, 'residual_energy': 4.1844370874139574e-05}, {'rows': 128, 'own_energy': 2.5699042734009483e-07, 'delta_energy': 2.0961759065481808e-07, 'residual_energy': 3.43377058244212e-05}, {'rows': 53, 'own_energy': 0.09963146042539041, 'delta_energy': 0.0629130676646082, 'residual_energy': 0.036386766256353384}, {'rows': 128, 'own_energy': 0.008781430913693291, 'delta_energy': 0.00555395646030972, 'residual_energy': 0.0013656155788557226}, {'rows': 128, 'own_energy': 6.12105199151946e-06, 'delta_energy': 1.6420915571635443e-05, 'residual_energy': 4.1951576340582266e-05}, {'rows': 128, 'own_energy': 1.9288175749588665e-05, 'delta_energy': 9.302478781870202e-06, 'residual_energy': 1.6996454683654532e-05}, {'rows': 52, 'own_energy': 0.13854843103191555, 'delta_energy': 0.1133569853750128, 'residual_energy': 0.04123827989997961}, {'rows': 128, 'own_energy': 0.01676888323475445, 'delta_energy': 0.00046831544061795203, 'residual_energy': 0.0001995241928766682}, {'rows': 128, 'own_energy': 0.0024541407696756527, 'delta_energy': 4.4172212980462344e-05, 'residual_energy': 1.9558819882785244e-05}, {'rows': 65, 'own_energy': 0.05097696890268316, 'delta_energy': 0.043903201362643855, 'residual_energy': 0.014460425096297103}, {'rows': 128, 'own_energy': 0.00021147147695817946, 'delta_energy': 0.00015347828249005391, 'residual_energy': 6.078409769732537e-05}, {'rows': 128, 'own_energy': 0.0011788104569673225, 'delta_energy': 2.2690753269417495e-05, 'residual_energy': 2.0849064177686836e-05}], 'projection': {'columns': 32, 'retained_columns': 32, 'fit_rows': 1657, 'test_rows': 1659}, 'scan_rows': 42919, 'valid_scan_rows': 42919, 'support': [True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True]}, {'client': 12, 'classes': [{'rows': 128, 'own_energy': 8.496396395602031e-06, 'delta_energy': 3.9935201511740114e-07, 'residual_energy': 1.5839565570347813e-05}, {'rows': 39, 'own_energy': 0.08695891239253328, 'delta_energy': 0.0638033966457974, 'residual_energy': 0.019649540077546712}, {'rows': 128, 'own_energy': 3.2088285042058736e-06, 'delta_energy': 4.189727212867472e-05, 'residual_energy': 2.1985825944644872e-05}, {'rows': 89, 'own_energy': 0.021393570285160944, 'delta_energy': 0.01607174784277241, 'residual_energy': 0.004510756749187418}, {'rows': 45, 'own_energy': 0.08295454723745839, 'delta_energy': 0.07297939546526874, 'residual_energy': 0.027546056112563545}, {'rows': 128, 'own_energy': 0.0008258996801810707, 'delta_energy': 0.000708286972977114, 'residual_energy': 0.0003844697027252539}, {'rows': 53, 'own_energy': 0.0966921313632514, 'delta_energy': 0.0536457585639618, 'residual_energy': 0.01802099562238636}, None, {'rows': 128, 'own_energy': 0.00022087325034977684, 'delta_energy': 3.27802743440025e-05, 'residual_energy': 2.185845047968004e-05}, {'rows': 128, 'own_energy': 9.482765797239768e-05, 'delta_energy': 1.4646719696860224e-06, 'residual_energy': 8.18386509042625e-06}, {'rows': 32, 'own_energy': 0.25308958466168746, 'delta_energy': 0.21746041363923305, 'residual_energy': 0.0775414852685341}, {'rows': 32, 'own_energy': 0.11203050955356789, 'delta_energy': 0.0573299216754234, 'residual_energy': 0.018963886078386387}, {'rows': 128, 'own_energy': 0.003504663360884026, 'delta_energy': 0.0001500142560702323, 'residual_energy': 3.816588827099664e-05}, {'rows': 128, 'own_energy': 6.459689038944684e-05, 'delta_energy': 0.00014991221698989072, 'residual_energy': 3.6617636381368465e-05}, None, {'rows': 128, 'own_energy': 1.716005173016278e-05, 'delta_energy': 6.741129681534603e-05, 'residual_energy': 4.0561185607427946e-05}], 'projection': {'columns': 32, 'retained_columns': 30, 'fit_rows': 1314, 'test_rows': 1314}, 'scan_rows': 34788, 'valid_scan_rows': 34788, 'support': [True, True, True, True, True, True, True, False, True, True, True, True, True, True, False, True]}, {'client': 13, 'classes': [{'rows': 128, 'own_energy': 0.0005518420826772074, 'delta_energy': 0.00033213009930992756, 'residual_energy': 7.873793729560081e-05}, {'rows': 128, 'own_energy': 0.017158277905220114, 'delta_energy': 0.006162255516368711, 'residual_energy': 0.0019189114859023938}, {'rows': 86, 'own_energy': 0.029037868902384033, 'delta_energy': 0.0233546809219642, 'residual_energy': 0.009066460034159838}, {'rows': 128, 'own_energy': 0.0014006610580828203, 'delta_energy': 0.0001531518823468285, 'residual_energy': 5.9442365982883245e-05}, {'rows': 128, 'own_energy': 2.3637167495143593e-05, 'delta_energy': 5.345768103129276e-05, 'residual_energy': 3.1595065206797025e-05}, {'rows': 128, 'own_energy': 1.06472288611236e-05, 'delta_energy': 5.018101508723551e-06, 'residual_energy': 3.832080862560438e-05}, {'rows': 52, 'own_energy': 0.1594428727317229, 'delta_energy': 0.1004392225927746, 'residual_energy': 0.032673005092746024}, {'rows': 79, 'own_energy': 0.02635908102175382, 'delta_energy': 0.021692609794752214, 'residual_energy': 0.005361706685918354}, {'rows': 128, 'own_energy': 1.3011045721982801e-05, 'delta_energy': 1.6213643385557967e-05, 'residual_energy': 3.582569360638978e-05}, {'rows': 128, 'own_energy': 1.1213453360372182e-05, 'delta_energy': 1.8926791537512263e-05, 'residual_energy': 3.698155721737233e-05}, {'rows': 128, 'own_energy': 0.002519029626366619, 'delta_energy': 0.0007593101866410524, 'residual_energy': 0.00032862317614071584}, {'rows': 36, 'own_energy': 0.08958999941271831, 'delta_energy': 0.04817003270510267, 'residual_energy': 0.01711097948987608}, {'rows': 128, 'own_energy': 0.0027131130695731663, 'delta_energy': 0.0003662685072748981, 'residual_energy': 9.263274962328135e-05}, {'rows': 128, 'own_energy': 0.0003151666491553217, 'delta_energy': 3.836744963272608e-05, 'residual_energy': 2.884962025185367e-05}, {'rows': 128, 'own_energy': 1.4314102545802138e-07, 'delta_energy': 3.3802641770285485e-08, 'residual_energy': 2.8055611484277928e-05}, {'rows': 46, 'own_energy': 0.19960908967105642, 'delta_energy': 0.16656258562482443, 'residual_energy': 0.044764550803565444}], 'projection': {'columns': 32, 'retained_columns': 32, 'fit_rows': 1704, 'test_rows': 1707}, 'scan_rows': 41827, 'valid_scan_rows': 41827, 'support': [True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True]}, {'client': 14, 'classes': [{'rows': 117, 'own_energy': 0.09682314761573929, 'delta_energy': 0.09315905699234758, 'residual_energy': 0.0061594731634893206}, {'rows': 36, 'own_energy': 0.9856622589738667, 'delta_energy': 0.8621444237045344, 'residual_energy': 0.1645584180071173}, {'rows': 128, 'own_energy': 0.03066140924732163, 'delta_energy': 0.02411627906912489, 'residual_energy': 0.008866408312177223}, {'rows': 128, 'own_energy': 0.005949171568748737, 'delta_energy': 0.0024055090927880387, 'residual_energy': 0.00039285032890614996}, {'rows': 89, 'own_energy': 0.10001370675028243, 'delta_energy': 0.08841309671736555, 'residual_energy': 0.023325924478081964}, {'rows': 128, 'own_energy': 1.8785096122472258e-11, 'delta_energy': 6.060920741739463e-12, 'residual_energy': 0.00012002651789262483}, {'rows': 128, 'own_energy': 0.0020295507440521747, 'delta_energy': 0.0003349620104045792, 'residual_energy': 0.0001417855258360559}, {'rows': 128, 'own_energy': 6.887631491092215e-06, 'delta_energy': 4.677004206174461e-05, 'residual_energy': 5.520701848835947e-05}, {'rows': 128, 'own_energy': 0.004569275218276943, 'delta_energy': 0.003284657444529543, 'residual_energy': 0.0013583471784401973}, {'rows': 128, 'own_energy': 0.00013834672454020964, 'delta_energy': 1.5044340720133446e-05, 'residual_energy': 4.417511612708196e-05}, {'rows': 58, 'own_energy': 0.29225532021260525, 'delta_energy': 0.2556847717673444, 'residual_energy': 0.09396737746295564}, {'rows': 128, 'own_energy': 0.012948888976098132, 'delta_energy': 8.609634502833771e-05, 'residual_energy': 5.874208162883835e-05}, {'rows': 128, 'own_energy': 0.018417026410365415, 'delta_energy': 0.01151922124347098, 'residual_energy': 0.0035651723666090054}, {'rows': 128, 'own_energy': 0.02530377124285433, 'delta_energy': 0.017741467631433035, 'residual_energy': 0.0034713021884829748}, {'rows': 128, 'own_energy': 2.8159653189400644e-08, 'delta_energy': 1.4443654220668075e-08, 'residual_energy': 7.603425517600707e-05}, {'rows': 38, 'own_energy': 0.5089478941104184, 'delta_energy': 0.46693945324270875, 'residual_energy': 0.07655330133640845}], 'projection': {'columns': 32, 'retained_columns': 32, 'fit_rows': 1743, 'test_rows': 1746}, 'scan_rows': 71183, 'valid_scan_rows': 71183, 'support': [True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True]}, {'client': 15, 'classes': [{'rows': 128, 'own_energy': 7.083751395227565e-05, 'delta_energy': 4.6997730694480465e-06, 'residual_energy': 5.059512797561741e-06}, {'rows': 128, 'own_energy': 0.0010385966312868968, 'delta_energy': 0.0001146269059880115, 'residual_energy': 2.24130411617909e-05}, {'rows': 87, 'own_energy': 0.02384423170703615, 'delta_energy': 0.017133061332386356, 'residual_energy': 0.0044456588025590105}, {'rows': 128, 'own_energy': 8.017593481540989e-06, 'delta_energy': 8.999774114951838e-05, 'residual_energy': 1.5784429896798047e-05}, {'rows': 128, 'own_energy': 0.00034413230516224956, 'delta_energy': 3.568554062043226e-05, 'residual_energy': 1.0669597208476833e-05}, {'rows': 128, 'own_energy': 0.00019404261852155644, 'delta_energy': 0.00015629993024496633, 'residual_energy': 3.748860478491902e-05}, {'rows': 128, 'own_energy': 0.00022353626881416197, 'delta_energy': 0.00020263743780647367, 'residual_energy': 1.71067893100907e-05}, {'rows': 22, 'own_energy': 0.15646659932109663, 'delta_energy': 0.1450310762695591, 'residual_energy': 0.043205739105989655}, {'rows': 128, 'own_energy': 3.5692555192569025e-05, 'delta_energy': 2.8452445602889767e-07, 'residual_energy': 4.844727341874305e-06}, {'rows': 128, 'own_energy': 0.0008990376344769369, 'delta_energy': 0.000293407262397601, 'residual_energy': 9.059281194158684e-05}, {'rows': 128, 'own_energy': 1.4871498652461008e-06, 'delta_energy': 4.874590033387299e-05, 'residual_energy': 2.4037227529629978e-05}, {'rows': 89, 'own_energy': 0.012147552146763113, 'delta_energy': 2.31770448121629e-05, 'residual_energy': 1.273283170260193e-05}, {'rows': 128, 'own_energy': 0.00010386392573596051, 'delta_energy': 0.0002703806068197707, 'residual_energy': 5.442032322572269e-05}, {'rows': 128, 'own_energy': 0.00016630103657617364, 'delta_energy': 0.00015256507711532794, 'residual_energy': 3.150398372894769e-05}, {'rows': 33, 'own_energy': 0.18421929232496712, 'delta_energy': 0.17987463683096472, 'residual_energy': 0.040601591097023604}, {'rows': 44, 'own_energy': 0.06514026456124943, 'delta_energy': 0.0500549097200101, 'residual_energy': 0.013990150650588092}], 'projection': {'columns': 32, 'retained_columns': 32, 'fit_rows': 1680, 'test_rows': 1683}, 'scan_rows': 28322, 'valid_scan_rows': 28322, 'support': [True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True]}, {'client': 16, 'classes': [{'rows': 128, 'own_energy': 0.000583936919243127, 'delta_energy': 0.0003469007801966508, 'residual_energy': 8.879303725553277e-05}, {'rows': 128, 'own_energy': 3.393462921684693e-05, 'delta_energy': 0.0006234974725126965, 'residual_energy': 0.0001843561918857879}, {'rows': 128, 'own_energy': 0.0003451664400255927, 'delta_energy': 6.6257479565978145e-06, 'residual_energy': 5.632696480299923e-06}, {'rows': 128, 'own_energy': 6.94816856772583e-05, 'delta_energy': 0.00019412612902789268, 'residual_energy': 4.5343644256376535e-05}, {'rows': 128, 'own_energy': 0.0002563200246007462, 'delta_energy': 5.8557737164402927e-05, 'residual_energy': 1.2563896327219076e-05}, {'rows': 128, 'own_energy': 1.7764336334468927e-05, 'delta_energy': 1.0578070471563978e-05, 'residual_energy': 1.1990780708738087e-05}, {'rows': 128, 'own_energy': 0.002790215620429237, 'delta_energy': 3.979953906263801e-05, 'residual_energy': 1.270628617434332e-05}, {'rows': 35, 'own_energy': 0.012110157680907337, 'delta_energy': 0.010097598204553226, 'residual_energy': 0.002481044542190847}, {'rows': 43, 'own_energy': 0.05070219476013736, 'delta_energy': 0.041052073699722136, 'residual_energy': 0.01253463524519517}, {'rows': 128, 'own_energy': 4.9684215042588724e-05, 'delta_energy': 1.821447987982428e-05, 'residual_energy': 8.911073854204836e-06}, {'rows': 128, 'own_energy': 0.0010165177916991526, 'delta_energy': 4.0490794500980134e-07, 'residual_energy': 2.0550459373305155e-06}, {'rows': 36, 'own_energy': 0.011211904220683735, 'delta_energy': 0.002252786334793742, 'residual_energy': 0.0011312782038523932}, {'rows': 128, 'own_energy': 3.1952361654469615e-08, 'delta_energy': 1.173252498119661e-06, 'residual_energy': 7.473563809098476e-06}, {'rows': 128, 'own_energy': 0.0002902612130084818, 'delta_energy': 9.438055692507791e-05, 'residual_energy': 2.33696851175607e-05}, {'rows': 33, 'own_energy': 0.09946040798516016, 'delta_energy': 0.09598316618080419, 'residual_energy': 0.025819462904884974}, {'rows': 50, 'own_energy': 0.03389554890252199, 'delta_energy': 0.02184740483189099, 'residual_energy': 0.004022214077816371}], 'projection': {'columns': 32, 'retained_columns': 32, 'fit_rows': 1602, 'test_rows': 1605}, 'scan_rows': 19882, 'valid_scan_rows': 19882, 'support': [True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True]}, {'client': 17, 'classes': [{'rows': 128, 'own_energy': 0.0227604487686593, 'delta_energy': 0.021038051366289653, 'residual_energy': 0.00667727389740993}, {'rows': 128, 'own_energy': 0.005724608480818603, 'delta_energy': 0.0014689966535956648, 'residual_energy': 0.00047381851499454363}, {'rows': 128, 'own_energy': 4.430574800353096e-05, 'delta_energy': 7.646114784718484e-07, 'residual_energy': 3.2211265522439654e-05}, {'rows': 99, 'own_energy': 0.32357911300689673, 'delta_energy': 0.29950873724687493, 'residual_energy': 0.10268202272871951}, {'rows': 128, 'own_energy': 0.0056158034321750275, 'delta_energy': 0.002827550762443178, 'residual_energy': 0.0009556693348651887}, {'rows': 128, 'own_energy': 7.524450508367542e-07, 'delta_energy': 1.1694952693901746e-08, 'residual_energy': 893.730879410005}, {'rows': 128, 'own_energy': 0.0010336818729757538, 'delta_energy': 0.00026443198753765037, 'residual_energy': 0.0002809858371648267}, {'rows': 128, 'own_energy': 0.0010803205562962938, 'delta_energy': 2.7834319276489373e-05, 'residual_energy': 3.108657576099381e-05}, {'rows': 128, 'own_energy': 0.0005260253501750438, 'delta_energy': 0.0002533667815336811, 'residual_energy': 0.0001570745064758917}, {'rows': 128, 'own_energy': 0.0010855794765852527, 'delta_energy': 0.000424554958774897, 'residual_energy': 0.00022844111127233536}, {'rows': 128, 'own_energy': 0.03351428088650096, 'delta_energy': 0.026352169943597077, 'residual_energy': 0.011863797918199966}, {'rows': 87, 'own_energy': 0.4195423914284638, 'delta_energy': 0.26963284134801385, 'residual_energy': 0.11670800116391837}, {'rows': 128, 'own_energy': 0.0036429918402221197, 'delta_energy': 0.0008832199738184321, 'residual_energy': 0.00030181078884500566}, {'rows': 128, 'own_energy': 0.0005585162387184455, 'delta_energy': 8.844907726981566e-06, 'residual_energy': 2.8742770611721858e-05}, {'rows': 128, 'own_energy': 6.491403111615509e-08, 'delta_energy': 5.57174776438623e-08, 'residual_energy': 0.00012010042148884111}, {'rows': 128, 'own_energy': 7.974009883921321e-05, 'delta_energy': 1.4541003568688587e-06, 'residual_energy': 2.726600346568336e-05}], 'projection': {'columns': 32, 'retained_columns': 32, 'fit_rows': 1978, 'test_rows': 1978}, 'scan_rows': 136358, 'valid_scan_rows': 136358, 'support': [True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True]}, {'client': 18, 'classes': [{'rows': 128, 'own_energy': 0.0029269242778839364, 'delta_energy': 0.0022803393058147712, 'residual_energy': 0.00021005672429353435}, {'rows': 76, 'own_energy': 0.017960759360581408, 'delta_energy': 0.006747585933077043, 'residual_energy': 0.0026374950492815573}, {'rows': 128, 'own_energy': 4.0292298255225035e-05, 'delta_energy': 2.55315759385926e-05, 'residual_energy': 8.263444439690153e-06}, {'rows': 77, 'own_energy': 0.014675112064471339, 'delta_energy': 0.010366419722155733, 'residual_energy': 0.0036102497308147037}, {'rows': 128, 'own_energy': 0.0016350215780651413, 'delta_energy': 0.00019412450496442016, 'residual_energy': 3.685428138310818e-05}, {'rows': 128, 'own_energy': 6.891407552476027e-05, 'delta_energy': 4.947221171011858e-05, 'residual_energy': 7.427028162510893e-05}, {'rows': 35, 'own_energy': 0.08647453565792694, 'delta_energy': 0.050493992480720035, 'residual_energy': 0.01779866483234426}, {'rows': 47, 'own_energy': 0.019929324875319503, 'delta_energy': 0.016437865355874923, 'residual_energy': 0.0035312505828150775}, {'rows': 128, 'own_energy': 0.0001586633113639248, 'delta_energy': 7.92400645843847e-06, 'residual_energy': 6.916240914463353e-06}, {'rows': 128, 'own_energy': 3.634957688827245e-06, 'delta_energy': 3.594623434672803e-05, 'residual_energy': 1.6633222355449078e-05}, {'rows': 128, 'own_energy': 8.1027617205946e-06, 'delta_energy': 9.145617109339572e-05, 'residual_energy': 4.657997543447773e-05}, {'rows': 128, 'own_energy': 0.0011221234131937167, 'delta_energy': 0.010944853179904113, 'residual_energy': 0.0012300363064653968}, {'rows': 128, 'own_energy': 2.2390717976569264e-05, 'delta_energy': 5.875633900296664e-05, 'residual_energy': 1.2640964671182336e-05}, {'rows': 128, 'own_energy': 0.006677808511188862, 'delta_energy': 0.003218220581066895, 'residual_energy': 0.0005500272320888638}, {'rows': 128, 'own_energy': 5.384379739445415e-07, 'delta_energy': 2.2801164282467268e-09, 'residual_energy': 3.341225872035818e-06}, {'rows': 128, 'own_energy': 5.12426280881081e-05, 'delta_energy': 0.00010074100273146721, 'residual_energy': 2.2941857612979743e-05}], 'projection': {'columns': 32, 'retained_columns': 32, 'fit_rows': 1770, 'test_rows': 1771}, 'scan_rows': 24804, 'valid_scan_rows': 24804, 'support': [True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True]}, {'client': 19, 'classes': [{'rows': 128, 'own_energy': 4.8072260176276624e-05, 'delta_energy': 8.9242000568999e-06, 'residual_energy': 3.4339948504752103e-06}, {'rows': 128, 'own_energy': 0.0014114845200200068, 'delta_energy': 4.572516518507588e-05, 'residual_energy': 1.1098777480156574e-05}, {'rows': 128, 'own_energy': 5.334139215367255e-05, 'delta_energy': 4.559282259521286e-05, 'residual_energy': 1.9895614580386923e-05}, {'rows': 128, 'own_energy': 0.001730540710793491, 'delta_energy': 0.00015699787695084815, 'residual_energy': 3.042288809681991e-05}, {'rows': 67, 'own_energy': 0.021444098158833105, 'delta_energy': 0.016729485437981244, 'residual_energy': 0.006316955115276418}, {'rows': 128, 'own_energy': 1.8041144599614565e-07, 'delta_energy': 1.761180066913988e-07, 'residual_energy': 1.0348796199708652e-05}, {'rows': 128, 'own_energy': 0.004890353748629872, 'delta_energy': 0.00014984725809032493, 'residual_energy': 3.390596857947333e-05}, {'rows': 80, 'own_energy': 0.004785248230715013, 'delta_energy': 0.0035152633427554696, 'residual_energy': 0.001045465578546847}, {'rows': 32, 'own_energy': 0.0968533213303047, 'delta_energy': 0.0864193338807234, 'residual_energy': 0.030509231658483675}, {'rows': 128, 'own_energy': 5.590312294647729e-07, 'delta_energy': 5.361196094292174e-06, 'residual_energy': 5.499694707202972e-06}, {'rows': 128, 'own_energy': 0.0010001034285144848, 'delta_energy': 1.4889553664837772e-05, 'residual_energy': 6.9693147244349395e-06}, {'rows': 39, 'own_energy': 0.016953333435620693, 'delta_energy': 0.005350711373379278, 'residual_energy': 0.0016494474510221906}, {'rows': 128, 'own_energy': 0.008877591149520563, 'delta_energy': 0.0021171615470775354, 'residual_energy': 0.00033124137024079963}, {'rows': 128, 'own_energy': 0.002577245236579738, 'delta_energy': 0.0009470689279304203, 'residual_energy': 0.00021339582079587448}, {'rows': 128, 'own_energy': 9.170705025796487e-05, 'delta_energy': 5.132731808147607e-05, 'residual_energy': 9.822913988506077e-06}, {'rows': 128, 'own_energy': 0.0006783806500233246, 'delta_energy': 3.878782867650634e-05, 'residual_energy': 9.423979974103306e-06}], 'projection': {'columns': 32, 'retained_columns': 32, 'fit_rows': 1752, 'test_rows': 1754}, 'scan_rows': 27151, 'valid_scan_rows': 27151, 'support': [True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True]}]}

assert hashlib.sha256(DIAGNOSTIC.encode()).hexdigest()==DIAGNOSTIC_SHA
(WORK/'proj/rank_projection_diagnostic.py').write_text(DIAGNOSTIC)
(SOURCE/'rank_projection_diagnostic.py').write_text(DIAGNOSTIC)
(SOURCE/'diagnostic_manifest.json').write_text(json.dumps(dict(sha256=DIAGNOSTIC_SHA,parent=REPLAY['parent'],audit_sha=REPLAY['audit_sha'])))
(OUT/'replay.json').write_text(json.dumps(REPLAY))


In [ ]:
SEED=42
SESSION_HOURS=.25

wb=wandb.init(project='mobihet-cross-client-rank-probe',id=f'ccr_diag_s{SEED}-{DIAGNOSTIC_SHA[:8]}',
    config=dict(seed=SEED,diagnostic_sha=DIAGNOSTIC_SHA,parent=REPLAY['parent'],no_training=True),mode='online',
    settings=wandb.Settings(save_code=False,disable_code=True))
assert not wb.settings._offline
active=[]
for shard in (0,1):
    assert time.monotonic()-SESSION_T0<SESSION_HOURS*3600-120,'budget exhausted before diagnostic'
    log=open(OUT/f'diagnostic_gpu{shard}.log','w')
    env=dict(os.environ,CUDA_VISIBLE_DEVICES=str(shard),PYTHONPATH=str(WORK),OMP_NUM_THREADS='1',MKL_NUM_THREADS='1',OPENBLAS_NUM_THREADS='1')
    p=subprocess.Popen([sys.executable,str(WORK/'proj/rank_projection_diagnostic.py'),'--input',str(SRC),
        '--out',str(OUT),'--seed',str(SEED),'--shard',str(shard)],cwd=WORK,env=env,stdout=log,stderr=subprocess.STDOUT)
    active.append((p,log))
while active:
    assert time.monotonic()-SESSION_T0<SESSION_HOURS*3600,'diagnostic budget exhausted'
    for p,log in list(active):
        if p.poll() is None:continue
        log.close();active.remove((p,log));assert p.returncode==0,('diagnostic worker failed',p.returncode)
    if active:time.sleep(2)
rows=[json.loads((OUT/f'diagnostic_{k:02d}.json').read_text()) for k in range(20)]
assert all(r['original_energies_reproduced'] for r in rows)
result=dict(seed=SEED,parent=REPLAY['parent'],all_original_energies_reproduced=True,models=20,
    diagnostic_sha=DIAGNOSTIC_SHA,model_training=False,holdout_read=False,gate_changed=False,
    elapsed_seconds=time.monotonic()-SESSION_T0)
(OUT/'diagnostic_result.json').write_text(json.dumps(result,indent=2))
wb.log(result);wb.finish()
print('DIAGNOSTIC COMPLETE',result,flush=True)
